# 🥩 YOLO Детекция Стейков - Оптимизированная Версия
### Обучение + Оптимизация гиперпараметров

**Ключевые особенности:**
- ✅ YOLOv11 (nano/small/medium/large)
- ✅ Optuna для подбора гиперпараметров
- ✅ Автоматический выбор лучшей модели
- ✅ Визуализация результатов
- ✅ Экспорт в ONNX/TorchScript

In [1]:
%matplotlib inline
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import os
import random
import yaml
from ultralytics import YOLO
import cv2
import shutil
import tempfile
import warnings
import optuna
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

e:\marbled_beef\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.7.1+cu118
CUDA: True
GPU: NVIDIA GeForce RTX 4070


In [2]:
# ====================== CONFIG ======================
class Config:
    # Данные
    data_dir = "all1"  # Папка с изображениями и JSON аннотациями
    output_dir = "yolo_data"
    
    # Модели для сравнения
    model_sizes = ['yolo11n.pt', 'yolo11s.pt']  # nano и small для скорости
    
    # Параметры обучения
    img_size = 640
    batch_size = 16
    epochs = 20  # Уменьшено для маленького датасета
    patience = 10  # Ранняя остановка
    
    # Optuna
    optuna_n_trials = 10
    optuna_max_epochs = 15  # Максимум эпох в Optuna
    
    # Device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed = 42

cfg = Config()

# Seed для воспроизводимости
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

# Глобальная переменная для config_path
config_path = None

In [3]:
# ====================== ЗАГРУЗКА АННОТАЦИЙ ======================
def load_labelme_annotations(data_dir):
    """Загрузка аннотаций LabelMe"""
    all_dir = Path(data_dir)
    json_files = list(all_dir.glob('*.json'))
    
    annotations = []
    for json_path in json_files:
        with open(json_path, 'r', encoding='utf-8') as f:
            ann = json.load(f)
        
        img_path = all_dir / ann['imagePath']
        if not img_path.exists():
            continue
        
        shapes = ann.get('shapes', [])
        if not shapes:
            continue
        
        annotations.append({
            'img_path': str(img_path),
            'shapes': shapes,
            'image_height': ann['imageHeight'],
            'image_width': ann['imageWidth']
        })
    
    if not annotations:
        return pd.DataFrame(), pd.DataFrame()
    
    ann_df = pd.DataFrame(annotations)
    
    # Разбиение на train/val
    train_indices = random.sample(range(len(ann_df)), int(len(ann_df) * 0.8))
    val_indices = [i for i in range(len(ann_df)) if i not in train_indices]
    
    return ann_df.iloc[train_indices].reset_index(drop=True), ann_df.iloc[val_indices].reset_index(drop=True)


def prepare_yolo_data(yolo_dir, train_ann, val_ann, class_to_idx):
    """Подготовка данных в формате YOLO"""
    for split, ann_df in [('train', train_ann), ('val', val_ann)]:
        split_dir = yolo_dir / split
        split_dir.mkdir(exist_ok=True)
        
        images_dir = split_dir / 'images'
        labels_dir = split_dir / 'labels'
        images_dir.mkdir(exist_ok=True)
        labels_dir.mkdir(exist_ok=True)
        
        for _, row in ann_df.iterrows():
            img_path = Path(row['img_path'])
            shutil.copy(img_path, images_dir / img_path.name)
            
            label_path = labels_dir / f"{img_path.stem}.txt"
            with open(label_path, 'w') as f:
                for shape in row['shapes']:
                    label = shape['label']
                    if label not in class_to_idx:
                        continue
                    
                    points = shape['points']
                    if len(points) < 3:
                        continue
                    
                    # Конвертация в полигон YOLO
                    normalized_points = []
                    for point in points:
                        if len(point) == 2:
                            x = point[0] / row['image_width']
                            y = point[1] / row['image_height']
                            normalized_points.extend([x, y])
                    
                    if not normalized_points:
                        continue
                    
                    class_id = class_to_idx[label] - 1
                    line = f"{class_id} " + " ".join([f"{p:.6f}" for p in normalized_points])
                    f.write(line + "\n")
    
    return yolo_dir


def create_yolo_config(yolo_dir, class_to_idx):
    """Создание конфигурационного файла YOLO"""
    config_data = {
        'path': str(yolo_dir),
        'train': 'train/images',
        'val': 'val/images',
        'nc': len(class_to_idx),
        'names': list(class_to_idx.keys())
    }
    
    config_path = yolo_dir / 'data.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config_data, f, default_flow_style=False)
    
    return config_path

In [4]:
# ====================== OPTUNA TRAINING ======================
def train_model_with_params(config_path, params, train_dir, device):
    """Обучение YOLO с заданными параметрами"""
    train_dir.mkdir(exist_ok=True)
    
    # Выбор модели
    model_size = params['model_size']
    model = YOLO(model_size)
    
    # Параметры обучения
    train_params = {
        'data': str(config_path),
        'epochs': params['epochs'],
        'imgsz': params['imgsz'],
        'batch': params['batch_size'],
        'patience': params['patience'],
        'device': device,
        'workers': 0,
        'seed': 42,
        'pretrained': True,
        'optimizer': params['optimizer'],
        'lr0': params['lr0'],
        'weight_decay': params['weight_decay'],
        'label_smoothing': params['label_smoothing'],
        'dropout': params['dropout'],
        'freeze': params['freeze_layers'],
        'save': True,
        'save_period': 0,
        'exist_ok': True,
        'project': str(train_dir),
        'name': 'train',
        'verbose': False,
    }
    
    # Аугментации
    if params['augmentation']:
        train_params.update({
            'augment': True,
            'hsv_h': params['hsv_h'],
            'hsv_s': params['hsv_s'],
            'hsv_v': params['hsv_v'],
            'degrees': params['degrees'],
            'translate': params['translate'],
            'scale': params['scale'],
            'fliplr': params['fliplr'],
        })
    else:
        train_params['augment'] = False
    
    # Обучение
    results = model.train(**train_params)
    
    best_model_path = train_dir / 'train' / 'weights' / 'best.pt'
    
    return best_model_path, results

In [5]:
# ====================== OPTUNA OBJECTIVE ======================
def objective(trial):
    """Функция для оптимизации гиперпараметров"""
    global config_path
    
    trial_number = trial.number + 1
    print(f"\n{'='*60}")
    print(f"OPTUNA TRIAL {trial_number}/{cfg.optuna_n_trials}")
    print(f"{'='*60}")
    
    # Гиперпараметры для оптимизации
    params = {
        'model_size': trial.suggest_categorical('model_size', ['yolo11n.pt', 'yolo11s.pt']),
        'epochs': trial.suggest_int('epochs', 10, cfg.optuna_max_epochs),  # Уменьшено для маленького датасета
        'imgsz': trial.suggest_categorical('imgsz', [416, 512, 640]),
        'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32]),
        'optimizer': trial.suggest_categorical('optimizer', ['SGD', 'Adam', 'AdamW']),
        'lr0': trial.suggest_loguniform('lr0', 1e-5, 1e-2),
        'weight_decay': trial.suggest_loguniform('weight_decay', 1e-5, 1e-2),
        'label_smoothing': trial.suggest_uniform('label_smoothing', 0.0, 0.2),
        'dropout': trial.suggest_uniform('dropout', 0.0, 0.5),
        'freeze_layers': trial.suggest_int('freeze_layers', 0, 10),
        'patience': trial.suggest_int('patience', 5, 15),  # Уменьшено
        'augmentation': trial.suggest_categorical('augmentation', [True, False]),
    }
    
    # Параметры аугментаций
    if params['augmentation']:
        params['hsv_h'] = trial.suggest_uniform('hsv_h', 0.0, 0.2)
        params['hsv_s'] = trial.suggest_uniform('hsv_s', 0.0, 0.2)
        params['hsv_v'] = trial.suggest_uniform('hsv_v', 0.0, 0.2)
        params['degrees'] = trial.suggest_uniform('degrees', 0.0, 45.0)
        params['translate'] = trial.suggest_uniform('translate', 0.0, 0.2)
        params['scale'] = trial.suggest_uniform('scale', 0.0, 0.2)
        params['fliplr'] = trial.suggest_uniform('fliplr', 0.0, 1.0)
    
    # Обучение
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_dir = Path(temp_dir)
        train_dir = temp_dir / 'train_results'
        
        try:
            best_model_path, results = train_model_with_params(
                config_path, params, train_dir, cfg.device
            )
            
            # Получаем метрики
            if hasattr(results, 'results_dict'):
                metrics = results.results_dict
                # Используем mAP50-95 как основную метрику
                val_metric = metrics.get('metrics/mAP50-95(B)', 0.0)
            else:
                val_metric = 0.0
            
            print(f"Trial {trial_number} completed. mAP50-95: {val_metric:.4f}")
            
        except Exception as e:
            print(f"Trial {trial_number} failed: {e}")
            val_metric = 0.0
    
    return val_metric

In [6]:
# ====================== MAIN ======================
def main():
    print(f"Device: {cfg.device}")
    print(f"Data directory: {cfg.data_dir}")
    print(f"Model sizes to try: {cfg.model_sizes}")
    
    # Классы для детекции
    class_to_idx = {
        'steak': 1,
    }
    classes = list(class_to_idx.keys())
    print(f"Classes: {classes}")
    
    # Загрузка аннотаций
    print("\nLoading annotations...")
    train_ann, val_ann = load_labelme_annotations(cfg.data_dir)
    
    if train_ann.empty or val_ann.empty:
        print("Error: No annotations found!")
        return
    
    print(f"Train samples: {len(train_ann)}")
    print(f"Val samples: {len(val_ann)}")
    
    # Подготовка данных YOLO
    print("\nPreparing YOLO data...")
    yolo_dir = Path(cfg.output_dir)
    yolo_dir.mkdir(exist_ok=True)
    
    yolo_dir = prepare_yolo_data(yolo_dir, train_ann, val_ann, class_to_idx)
    global config_path
    config_path = create_yolo_config(yolo_dir, class_to_idx)
    
    print(f"YOLO data prepared in: {yolo_dir}")
    print(f"Config saved to: {config_path}")
    
    # Optuna оптимизация
    print("\n" + "="*60)
    print("STARTING HYPERPARAMETER OPTIMIZATION")
    print(f"Number of trials: {cfg.optuna_n_trials}")
    print("="*60)
    
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=cfg.seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
    )
    
    study.optimize(objective, n_trials=cfg.optuna_n_trials)
    
    # Результаты
    print(f"\n{'='*60}")
    print("OPTIMIZATION COMPLETED!")
    print(f"{'='*60}")
    
    print(f"\nBest trial:")
    print(f"  Value (mAP50-95): {study.best_value:.4f}")
    print("  Best hyperparameters:")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")
    
    # Сохранение результатов
    df = study.trials_dataframe()
    df.to_csv("yolo_optuna_results.csv", index=False)
    print(f"\nResults saved to 'yolo_optuna_results.csv'")
    
    # Сохранение лучших параметров
    best_params = study.best_params
    with open('best_yolo_params.yaml', 'w') as f:
        yaml.dump(best_params, f)
    print(f"Best parameters saved to 'best_yolo_params.yaml'")
    
    # Финальное обучение с лучшими параметрами
    print("\n" + "="*60)
    print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
    print("="*60)
    
    final_train_dir = Path('yolo_final_model')
    best_model_path, final_results = train_model_with_params(
        config_path, best_params, final_train_dir, cfg.device
    )
    
    # Копирование лучшей модели
    if best_model_path.exists():
        final_model_path = Path('best_steak_detector.pt')
        shutil.copy(best_model_path, final_model_path)
        print(f"\n✅ Best model saved as 'best_steak_detector.pt'")
    
    # Визуализация метрик
    print("\nPlotting training metrics...")
    plot_training_metrics(final_train_dir / 'train')
    
    # Тестирование на валидации
    print("\nRunning validation...")
    final_model = YOLO(str(final_model_path))
    val_results = final_model.val(
        data=str(config_path),
        split='val',
        verbose=True
    )
    
    print(f"\n{'='*60}")
    print("FINAL RESULTS")
    print(f"{'='*60}")
    print(f"mAP50-95: {val_results.box.map50-95:.4f}")
    print(f"mAP50: {val_results.box.map50:.4f}")
    print(f"mAP75: {val_results.box.map75:.4f}")
    print(f"Precision: {val_results.box.mp:.4f}")
    print(f"Recall: {val_results.box.mr:.4f}")


def plot_training_metrics(results_path):
    """Визуализация метрик обучения"""
    csv_path = results_path / 'results.csv'
    if not csv_path.exists():
        return
    
    df = pd.read_csv(csv_path)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Training Metrics', fontsize=14, fontweight='bold')
    
    # Loss
    ax1 = axes[0, 0]
    if 'train/box_loss' in df.columns:
        ax1.plot(df['train/box_loss'], label='Box Loss', linewidth=2)
    if 'val/box_loss' in df.columns:
        ax1.plot(df['val/box_loss'], label='Val Box Loss', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Box Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Precision & Recall
    ax2 = axes[0, 1]
    if 'metrics/precision(B)' in df.columns:
        ax2.plot(df['metrics/precision(B)'], label='Precision', linewidth=2)
    if 'metrics/recall(B)' in df.columns:
        ax2.plot(df['metrics/recall(B)'], label='Recall', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Score')
    ax2.set_title('Precision & Recall')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # mAP
    ax3 = axes[1, 0]
    if 'metrics/mAP50(B)' in df.columns:
        ax3.plot(df['metrics/mAP50(B)'], label='mAP@0.5', linewidth=2, color='green')
    if 'metrics/mAP50-95(B)' in df.columns:
        ax3.plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', linewidth=2, color='blue')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('mAP')
    ax3.set_title('mAP Metrics')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Learning Rate
    ax4 = axes[1, 1]
    if 'lr/pg0' in df.columns:
        ax4.plot(df['lr/pg0'], label='Learning Rate', linewidth=2, color='purple')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Learning Rate')
        ax4.set_title('Learning Rate')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('yolo_training_metrics.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Training metrics saved to 'yolo_training_metrics.png'")


if __name__ == "__main__":
    main()

Device: cuda
Data directory: all1
Model sizes to try: ['yolo11n.pt', 'yolo11s.pt']
Classes: ['steak']

Loading annotations...
Train samples: 238
Val samples: 60

Preparing YOLO data...


[I 2026-02-27 12:57:54,261] A new study created in memory with name: no-name-f8836495-cea2-4a51-b346-6afbdaf10763


YOLO data prepared in: yolo_data
Config saved to: yolo_data\data.yaml

STARTING HYPERPARAMETER OPTIMIZATION
Number of trials: 10

OPTUNA TRIAL 1/10
New https://pypi.org/project/ultralytics/8.4.18 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.235  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data\data.yaml, degrees=16.486282948216125, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.09170225492671691, dynamic=False, embed=None, epochs=14, erasing=0.4, exist_ok=True, fliplr=0.19967378215835974, flipud=0.0, format=torchscript, fraction=1.0, freeze=3, half=False, hsv_h=0.1223705789

[I 2026-02-27 12:59:07,013] Trial 0 finished with value: 0.9175393790464728 and parameters: {'model_size': 'yolo11s.pt', 'epochs': 14, 'imgsz': 416, 'batch_size': 16, 'optimizer': 'AdamW', 'lr0': 0.00314288089084011, 'weight_decay': 4.335281794951564e-05, 'label_smoothing': 0.03636499344142013, 'dropout': 0.09170225492671691, 'freeze_layers': 3, 'patience': 10, 'augmentation': True, 'hsv_h': 0.1223705789444759, 'hsv_s': 0.027898772130408367, 'hsv_v': 0.058428929707043636, 'degrees': 16.486282948216125, 'translate': 0.0912139968434072, 'scale': 0.15703519227860274, 'fliplr': 0.19967378215835974}. Best is trial 0 with value: 0.9175393790464728.


Trial 1 completed. mAP50-95: 0.9175

OPTUNA TRIAL 2/10
New https://pypi.org/project/ultralytics/8.4.18 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.235  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data\data.yaml, degrees=43.63130824940514, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.017194260557609198, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.8948273504276488, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.10401360423556216, hsv_s=0.10934205586865593, hsv_v=0.03697089110510541, imgsz=416, int8=False, iou=0.7

...

Trial 8 completed. mAP50-95: 0.9636

OPTUNA TRIAL 9/10
New https://pypi.org/project/ultralytics/8.4.18 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.235  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data\data.yaml, degrees=10.97953395205876, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.37324570255901207, dynamic=False, embed=None, epochs=13, erasing=0.4, exist_ok=True, fliplr=0.8920465551771133, flipud=0.0, format=torchscript, fraction=1.0, freeze=7, half=False, hsv_h=0.018734953565618495, hsv_s=0.0735431606118867, hsv_v=0.05304047353634509, imgsz=640, int8=False, iou=0.7, 

[I 2026-02-27 13:09:02,794] Trial 8 finished with value: 0.9745819399589939 and parameters: {'model_size': 'yolo11s.pt', 'epochs': 13, 'imgsz': 640, 'batch_size': 32, 'optimizer': 'SGD', 'lr0': 0.001369423146002436, 'weight_decay': 5.149288946949013e-05, 'label_smoothing': 0.06507993963185355, 'dropout': 0.37324570255901207, 'freeze_layers': 7, 'patience': 14, 'augmentation': True, 'hsv_h': 0.018734953565618495, 'hsv_s': 0.0735431606118867, 'hsv_v': 0.05304047353634509, 'degrees': 10.97953395205876, 'translate': 0.19460211095048913, 'scale': 0.07861954493335209, 'fliplr': 0.8920465551771133}. Best is trial 2 with value: 0.98503320957866.


Trial 9 completed. mAP50-95: 0.9746

OPTUNA TRIAL 10/10
New https://pypi.org/project/ultralytics/8.4.18 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.235  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data\data.yaml, degrees=38.30115021825856, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.007728308264433714, dynamic=False, embed=None, epochs=13, erasing=0.4, exist_ok=True, fliplr=0.5568012624583502, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.170601891093472, hsv_s=0.05888977841391714, hsv_v=0.07701954572038505, imgsz=416, int8=False, iou=0.7, 

[I 2026-02-27 13:10:41,933] Trial 9 finished with value: 0.8949687247289345 and parameters: {'model_size': 'yolo11s.pt', 'epochs': 13, 'imgsz': 416, 'batch_size': 8, 'optimizer': 'AdamW', 'lr0': 0.00727420826493834, 'weight_decay': 0.005553837526912238, 'label_smoothing': 0.07403174005108888, 'dropout': 0.007728308264433714, 'freeze_layers': 10, 'patience': 9, 'augmentation': True, 'hsv_h': 0.170601891093472, 'hsv_s': 0.05888977841391714, 'hsv_v': 0.07701954572038505, 'degrees': 38.30115021825856, 'translate': 0.06338440103125553, 'scale': 0.033898549337218496, 'fliplr': 0.5568012624583502}. Best is trial 2 with value: 0.98503320957866.


Trial 10 completed. mAP50-95: 0.8950

OPTIMIZATION COMPLETED!

Best trial:
  Value (mAP50-95): 0.9850
  Best hyperparameters:
    model_size: yolo11s.pt
    epochs: 10
    imgsz: 640
    batch_size: 32
    optimizer: AdamW
    lr0: 2.6471141828218167e-05
    weight_decay: 0.002550298070162891
    label_smoothing: 0.014910128735954166
    dropout: 0.49344346830025865
    freeze_layers: 8
    patience: 7
    augmentation: False

Results saved to 'yolo_optuna_results.csv'
Best parameters saved to 'best_yolo_params.yaml'

TRAINING FINAL MODEL WITH BEST PARAMETERS
New https://pypi.org/project/ultralytics/8.4.18 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.235  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosa